# Лабораторная работа №1


In [12]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


plt.rcParams.update({"figure.figsize": (8, 4), "figure.dpi": 120})
pd.set_option("display.max_columns", 20)


In [13]:
data_dir = Path("data")

bike_path = data_dir / "day.csv"
spaceship_path = data_dir / "train.csv"

display(pd.read_csv(bike_path).head())
display(pd.read_csv(spaceship_path).head())


,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,6,0,2,0.344167,0.363625,0.805833,0.160446,331,654,985
1,2,2011-01-02,1,0,1,0,0,0,2,0.363478,0.353739,0.696087,0.248539,131,670,801
2,3,2011-01-03,1,0,1,0,1,1,1,0.196364,0.189405,0.437273,0.248309,120,1229,1349
3,4,2011-01-04,1,0,1,0,2,1,1,0.200000,0.212122,0.590435,0.160296,108,1454,1562
4,5,2011-01-05,1,0,1,0,3,1,1,0.226957,0.229270,0.436957,0.186900,82,1518,1600


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [9]:
def train_test_indices(sample_count, test_fraction=0.2, seed=8055):
    generator = np.random.default_rng(seed)
    indices = generator.permutation(sample_count)
    test_count = int(np.ceil(sample_count * test_fraction))
    return indices[test_count:], indices[:test_count]


## Предобработка
Медианы и параметры стандартизации считаются по train. Категории кодируются бинарными столбцами; первая категория опорная, неизвестная кодируется нулями.


In [ ]:
def prepare_features(train_features, test_features, numeric_columns, categorical_columns):
    medians = train_features[numeric_columns].median().fillna(0.0)
    train_numeric = train_features[numeric_columns].fillna(medians).to_numpy(dtype=float)
    test_numeric = test_features[numeric_columns].fillna(medians).to_numpy(dtype=float)

    means = np.mean(train_numeric, axis=0)
    deviations = np.std(train_numeric, axis=0)
    deviations[deviations == 0] = 1.0

    train_columns = [(train_numeric - means) / deviations]
    test_columns = [(test_numeric - means) / deviations]

    for column in categorical_columns:
        train_values = train_features[column].astype("string").fillna("Unknown")
        test_values = test_features[column].astype("string").fillna("Unknown")

        for category in sorted(train_values.unique())[1:]:
            train_columns.append((train_values == category).to_numpy(dtype=float))
            test_columns.append((test_values == category).to_numpy(dtype=float))

    return np.column_stack(train_columns), np.column_stack(test_columns)


## Метрики


In [ ]:
def mean_absolute_error(targets, predictions):
    return np.mean(np.abs(predictions - targets))


def mean_squared_error(targets, predictions):
    return np.mean((predictions - targets) ** 2)


def root_mean_squared_error(targets, predictions):
    return mean_squared_error(targets, predictions) ** 0.5


def coefficient_of_determination(targets, predictions):
    residual_sum = np.sum((targets - predictions) ** 2)
    total_sum = np.sum((targets - np.mean(targets)) ** 2)
    return 1 - residual_sum / total_sum if total_sum != 0 else np.nan


def confusion_matrix(targets, predictions):
    true_positive = np.sum((targets == 1) & (predictions == 1))
    false_positive = np.sum((targets == 0) & (predictions == 1))
    false_negative = np.sum((targets == 1) & (predictions == 0))
    true_negative = np.sum((targets == 0) & (predictions == 0))
    return np.array([[true_negative, false_positive], [false_negative, true_positive]])


def accuracy(targets, predictions):
    return np.mean(targets == predictions)


def precision(targets, predictions):
    _, false_positive, _, true_positive = confusion_matrix(targets, predictions).ravel()
    predicted_positive = true_positive + false_positive
    return true_positive / predicted_positive if predicted_positive else 0.0


def recall(targets, predictions):
    _, _, false_negative, true_positive = confusion_matrix(targets, predictions).ravel()
    actual_positive = true_positive + false_negative
    return true_positive / actual_positive if actual_positive else 0.0


def f1_score(targets, predictions):
    precision_value = precision(targets, predictions)
    recall_value = recall(targets, predictions)
    denominator = precision_value + recall_value
    return 2 * precision_value * recall_value / denominator if denominator else 0.0


def log_loss(targets, probabilities):
    epsilon = np.finfo(float).eps
    probabilities = np.clip(probabilities, epsilon, 1 - epsilon)
    return -np.mean(
        targets * np.log(probabilities)
        + (1 - targets) * np.log1p(-probabilities)
    )


## Модели
Полный градиентный спуск. В историю записываются потери после обновления параметров.


In [ ]:
def sigmoid(values):
    result = np.empty_like(values, dtype=float)
    positive = values >= 0
    result[positive] = 1 / (1 + np.exp(-values[positive]))
    exponent = np.exp(values[~positive])
    result[~positive] = exponent / (1 + exponent)
    return result


In [ ]:
class LinearRegression:
    def __init__(self, learning_rate=0.03, epochs=8000):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.weights = None
        self.bias = 0.0
        self.history = []

    def fit(self, features, targets):
        self.weights = np.zeros(features.shape[1])
        self.bias = 0.0
        self.history = []

        for epoch in range(self.epochs):
            predictions = features @ self.weights + self.bias
            errors = predictions - targets
            weights_gradient = 2 / len(targets) * (features.T @ errors)
            bias_gradient = 2 * np.mean(errors)

            self.weights -= self.learning_rate * weights_gradient
            self.bias -= self.learning_rate * bias_gradient

            predictions = features @ self.weights + self.bias
            self.history.append(mean_squared_error(targets, predictions))

        return self

    def predict(self, features):
        if self.weights is None:
            raise ValueError("Модель не обучалась")

        return features @ self.weights + self.bias


class LogisticRegression:
    def __init__(self, learning_rate=0.1, epochs=8000, threshold=0.5):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.threshold = threshold
        self.weights = None
        self.bias = 0.0
        self.history = []

    def fit(self, features, targets):
        self.weights = np.zeros(features.shape[1])
        self.bias = 0.0
        self.history = []

        for epoch in range(self.epochs):
            logits = features @ self.weights + self.bias
            probabilities = sigmoid(logits)
            errors = probabilities - targets
            weights_gradient = features.T @ errors / len(targets)
            bias_gradient = np.mean(errors)

            self.weights -= self.learning_rate * weights_gradient
            self.bias -= self.learning_rate * bias_gradient

            logits = features @ self.weights + self.bias
            self.history.append(np.mean(np.logaddexp(0, logits) - targets * logits))

        return self

    def predict_proba(self, features):
        if self.weights is None:
            raise ValueError("Модель не обучалась")

        return sigmoid(features @ self.weights + self.bias)

    def predict(self, features):
        return (self.predict_proba(features) >= self.threshold).astype(int)


## A. Linear Regression
Цель -- `cnt`. Из признаков исключены `casual` и `registered`, поскольку их сумма равна `cnt`; также исключены индекс `instant` и строка даты `dteday`. Календарные признаки сохранены отдельно. Проверка проводится на случайно отложенных днях.


In [ ]:
bike_data = pd.read_csv("day.csv")

print("Размер:", bike_data.shape)
print("Дубликатов:", bike_data.duplicated().sum())
display(bike_data.head())
display(pd.DataFrame({
    "Тип": bike_data.dtypes.astype(str),
    "Пропуски": bike_data.isna().sum(),
    "Уникальных": bike_data.nunique(),
}))
display(bike_data[["cnt", "temp", "hum", "windspeed"]].describe().round(3))


In [ ]:
bike_features = bike_data.drop(
    columns=["cnt", "casual", "registered", "instant", "dteday"]
)
bike_targets = bike_data["cnt"].to_numpy(dtype=float)

bike_numeric_columns = [
    "yr", "holiday", "workingday", "temp", "atemp", "hum", "windspeed"
]
bike_categorical_columns = ["season", "mnth", "weekday", "weathersit"]

bike_train_indices, bike_test_indices = train_test_indices(len(bike_data))
bike_train_targets = bike_targets[bike_train_indices]
bike_test_targets = bike_targets[bike_test_indices]

bike_train_features, bike_test_features = prepare_features(
    bike_features.iloc[bike_train_indices],
    bike_features.iloc[bike_test_indices],
    bike_numeric_columns,
    bike_categorical_columns,
)

print("Train:", bike_train_features.shape, "Test:", bike_test_features.shape)


In [ ]:
linear_model = LinearRegression(learning_rate=0.03, epochs=8000)
linear_model.fit(bike_train_features, bike_train_targets)

print("MSE после первой эпохи:", round(linear_model.history[0], 3))
print("MSE после последней эпохи:", round(linear_model.history[-1], 3))

plt.figure()
plt.plot(np.arange(1, len(linear_model.history) + 1), linear_model.history)
plt.xlabel("Эпоха")
plt.ylabel("MSE")
plt.title("Linear Regression")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
bike_predictions = linear_model.predict(bike_test_features)
bike_baseline_predictions = np.full(
    len(bike_test_targets), np.mean(bike_train_targets)
)

bike_results = []
for name, predictions in [
    ("LinearRegression", bike_predictions),
    ("Baseline", bike_baseline_predictions),
]:
    bike_results.append({
        "Модель": name,
        "MAE": mean_absolute_error(bike_test_targets, predictions),
        "RMSE": root_mean_squared_error(bike_test_targets, predictions),
        "R^2": coefficient_of_determination(bike_test_targets, predictions),
    })

bike_results = pd.DataFrame(bike_results).set_index("Модель")
display(bike_results.round(3))


In [ ]:
plt.figure()
plt.scatter(bike_test_targets, bike_predictions, alpha=0.65, s=20)
bike_minimum = min(bike_test_targets.min(), bike_predictions.min())
bike_maximum = max(bike_test_targets.max(), bike_predictions.max())
plt.plot([bike_minimum, bike_maximum], [bike_minimum, bike_maximum], color="tab:red")
plt.xlabel("Истинное cnt")
plt.ylabel("Предсказанное cnt")
plt.title("Bike Sharing: предсказания на test")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


**Результат.** На test: MAE = 560.98, RMSE = 732.11, R² = 0.839. У baseline RMSE = 1823.12, R² = -0.001. Модель улучшила все три метрики относительно baseline.

## B. Logistic Regression
Используется `train.csv`. Цель -- `Transported`: `False -> 0`, `True -> 1`. `PassengerId` и `Name` исключены; из `Cabin` выделены палуба и сторона. Порог -- 0.5.


In [ ]:
spaceship_data = pd.read_csv("train.csv")

print("Размер:", spaceship_data.shape)
print("Дубликатов:", spaceship_data.duplicated().sum())
display(spaceship_data.head())
display(pd.DataFrame({
    "Тип": spaceship_data.dtypes.astype(str),
    "Пропуски": spaceship_data.isna().sum(),
    "Уникальных": spaceship_data.nunique(),
}))
display(spaceship_data["Transported"].value_counts().to_frame("Количество"))
display(spaceship_data[["Age", "RoomService", "FoodCourt", "Spa"]].describe().round(3))


In [ ]:
spaceship_features_data = spaceship_data.drop(
    columns=["Transported", "PassengerId", "Name", "Cabin"]
)
cabin = spaceship_data["Cabin"].str.split("/", expand=True)
spaceship_features_data["Deck"] = cabin[0]
spaceship_features_data["Side"] = cabin[2]
spaceship_targets = spaceship_data["Transported"].to_numpy(dtype=int)

spaceship_numeric_columns = [
    "Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"
]
spaceship_categorical_columns = [
    "HomePlanet", "CryoSleep", "Destination", "VIP", "Deck", "Side"
]

spaceship_train_indices, spaceship_test_indices = train_test_indices(len(spaceship_data))
spaceship_train_targets = spaceship_targets[spaceship_train_indices]
spaceship_test_targets = spaceship_targets[spaceship_test_indices]

spaceship_train_features, spaceship_test_features = prepare_features(
    spaceship_features_data.iloc[spaceship_train_indices],
    spaceship_features_data.iloc[spaceship_test_indices],
    spaceship_numeric_columns,
    spaceship_categorical_columns,
)

display(pd.DataFrame({
    "Объектов": [len(spaceship_train_targets), len(spaceship_test_targets)],
    "Признаков": [spaceship_train_features.shape[1], spaceship_test_features.shape[1]],
    "Доля класса 1": [np.mean(spaceship_train_targets), np.mean(spaceship_test_targets)],
}, index=["Train", "Test"]).round(3))


In [ ]:
logistic_model = LogisticRegression(learning_rate=0.1, epochs=8000, threshold=0.5)
logistic_model.fit(spaceship_train_features, spaceship_train_targets)

print("BCE после первой эпохи:", round(logistic_model.history[0], 6))
print("BCE после последней эпохи:", round(logistic_model.history[-1], 6))

plt.figure()
plt.plot(np.arange(1, len(logistic_model.history) + 1), logistic_model.history)
plt.xlabel("Эпоха")
plt.ylabel("BCE")
plt.title("Logistic Regression")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
spaceship_probabilities = logistic_model.predict_proba(spaceship_test_features)
spaceship_predictions = logistic_model.predict(spaceship_test_features)

spaceship_baseline_probability = np.mean(spaceship_train_targets)
spaceship_baseline_class = int(spaceship_baseline_probability >= 0.5)
spaceship_baseline_predictions = np.full(
    len(spaceship_test_targets), spaceship_baseline_class
)
spaceship_baseline_probabilities = np.full(
    len(spaceship_test_targets), spaceship_baseline_probability
)

spaceship_results = []
for name, predictions, probabilities in [
    ("LogisticRegression", spaceship_predictions, spaceship_probabilities),
    ("Majority baseline", spaceship_baseline_predictions, spaceship_baseline_probabilities),
]:
    spaceship_results.append({
        "Модель": name,
        "Accuracy": accuracy(spaceship_test_targets, predictions),
        "Precision": precision(spaceship_test_targets, predictions),
        "Recall": recall(spaceship_test_targets, predictions),
        "F1": f1_score(spaceship_test_targets, predictions),
        "LogLoss": log_loss(spaceship_test_targets, probabilities),
    })

spaceship_results = pd.DataFrame(spaceship_results).set_index("Модель")
display(spaceship_results.round(4))


In [ ]:
spaceship_confusion_matrix = confusion_matrix(
    spaceship_test_targets, spaceship_predictions
)
display(pd.DataFrame(
    spaceship_confusion_matrix,
    index=["Истинный 0", "Истинный 1"],
    columns=["Прогноз 0", "Прогноз 1"],
))

plt.figure(figsize=(5, 4))
plt.imshow(spaceship_confusion_matrix, cmap="Blues")
for row in range(2):
    for column in range(2):
        value = spaceship_confusion_matrix[row, column]
        color = "white" if value > spaceship_confusion_matrix.max() / 2 else "black"
        plt.text(
            column, row, str(value),
            ha="center", va="center", fontsize=14, color=color,
        )
plt.xticks([0, 1], ["0", "1"])
plt.yticks([0, 1], ["0", "1"])
plt.xlabel("Предсказанный класс")
plt.ylabel("Истинный класс")
plt.title("Spaceship Titanic: матрица ошибок")
plt.tight_layout()
plt.show()


**Результат.** На test: Accuracy = 0.791, Precision = 0.771, Recall = 0.820, F1 = 0.795, LogLoss = 0.450. У majority baseline Accuracy = 0.494, LogLoss = 0.693. Recall ниже baseline, который всегда предсказывает класс 1.

## Вывод
Обе модели улучшили основные метрики относительно baseline. Параметры предобработки вычислялись только по train, настройки и порог заданы заранее. Для дальнейшей проверки можно выделить validation и подобрать на ней настройки.
